# only capture the frame from webcam

In [ ]:
import cv2
#read video
cap = cv2.VideoCapture(0)

mirror = True
#mirror the video


while True:
    ret, frame = cap.read()
    if not ret:
        break
    if mirror:
        frame = cv2.flip(frame, 1)
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

# draw rectangle on face

In [1]:
import cv2
cap = cv2.VideoCapture(0)
mirror = True
#crop face using harcascade
face_cascade = cv2.CascadeClassifier(r'C:\Users\LENOVO\OneDrive\Documents\1PYTHON\7_AI_ML\haarcascade_frontalface_default.xml')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if mirror:
        frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x,y,w,h) in faces:
        cv2.rectangle(frame, (x,y), (x+w, y+h), (255,0,0), 2)
        #cover the face with a circle
        # cv2.circle(frame, (x+w//2, y+h//2), w//2, (255,0,0), -1)
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()


In [3]:
!pip install opencv-contrib-python

!pip install opencv-contrib-python

# Train

In [5]:
# train model to recognize face using LBPH
import os
import numpy as np
from PIL import Image

# reuse existing face_cascade and recognizer objects if available
recognizer = cv2.face.LBPHFaceRecognizer_create()
save_dir = r'C:\Users\LENOVO\OneDrive\Documents\1PYTHON\7_AI_ML\dataset'  # update this path as needed

def get_images_and_labels(base_path):
    face_samples = []
    ids = []

    for name in os.listdir(base_path):
        path = os.path.join(base_path, name)

        # Case 1: files like User.<id>.<count>.jpg (your current format)
        if os.path.isfile(path) and name.lower().endswith((".jpg", ".jpeg", ".png")):
            parts = name.split(".")
            if len(parts) >= 3 and parts[0].lower() == "user" and parts[1].isdigit():
                user_id = int(parts[1])
                img = Image.open(path).convert("L")
                img_np = np.array(img, dtype="uint8")

                # Images are already cropped faces; use full image directly
                face_samples.append(img_np)
                ids.append(user_id)

        # Case 2: optional support for folder-per-person datasets
        elif os.path.isdir(path):
            # if needed, add folder parsing here
            pass

    return face_samples, ids

base_path = save_dir  # already defined in your notebook
faces, ids = get_images_and_labels(base_path)

print("Found samples:", len(faces))
print("Unique IDs:", sorted(set(ids)) if ids else [])

if len(faces) < 100:
    raise ValueError("Not enough training samples found. Capture more face images first.")

recognizer.train(faces, np.array(ids, dtype=np.int32))
recognizer.save(r'C:\Users\LENOVO\OneDrive\Documents\1PYTHON\7_AI_ML\trainer1.yml')
print("Training complete. Faces trained:", len(faces))


Found samples: 100
Unique IDs: [2]
Training complete. Faces trained: 100


# run

In [6]:
#recognize face using LBPH
import cv2
import numpy as np
mirror = True
face_cascade = cv2.CascadeClassifier(r'C:\Users\LENOVO\OneDrive\Documents\1PYTHON\7_AI_ML\haarcascade_frontalface_default.xml')
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read(r'C:\Users\LENOVO\OneDrive\Documents\1PYTHON\7_AI_ML\trainer1.yml')
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if mirror:
        frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    for (x,y,w,h) in faces:
        id, confidence = recognizer.predict(gray[y:y+h, x:x+w])
        if confidence < 100:
            id = "User " + str(id)
            confidence = "  {0}%".format(round(100 - confidence))
        else:
            id = "Unknown"
            confidence = "  {0}%".format(round(100 - confidence))
        cv2.rectangle(frame, (x,y), (x+w, y+h), (255,0,0), 2) 
        cv2.putText(frame, str(id), (x+5,y-5), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        cv2.putText(frame, str(confidence), (x+5,y+h-5), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 1)
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()


# Capture images

In [4]:
# capture 100 images from camera and save
save_dir = r'C:\Users\LENOVO\OneDrive\Documents\1PYTHON\7_AI_ML\dataset'  # fixed path
user_id = 2
count = 0
mirror = True

cap = cv2.VideoCapture(0)
face_cascade = cv2.CascadeClassifier(r'C:\Users\LENOVO\OneDrive\Documents\1PYTHON\7_AI_ML\haarcascade_frontalface_default.xml')

while True:
    ret, frame = cap.read()
    if not ret or count >= 100:
        break

    if mirror:
        frame = cv2.flip(frame, 1)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        count += 1
        file_path = f"{save_dir}\\User.{user_id}.{count}.jpg"
        ok = cv2.imwrite(file_path, gray[y:y+h, x:x+w])
        if not ok:
            print("Failed to save:", file_path)

        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Saved images:", count)


Saved images: 100
